# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah-9862873/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook writes a data contract for the Refresh / Content Opportunity Scoring lane, proves three facts with warehouse queries, builds five features, and runs the deliberate leakage experiment.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

In [5]:
%pip install -q duckdb pandas python-dotenv

import duckdb, os
from dotenv import load_dotenv

# Load .env file if it exists
load_dotenv()

hf_token = os.environ.get('HF_TOKEN') or None
if hf_token:
    print('HF token loaded.')
else:
    print('No HF_TOKEN found. Create a .env file with HF_TOKEN=your-token.')

# Connect to DuckDB and install/load httpfs
con = duckdb.connect()
con.execute('INSTALL httpfs; LOAD httpfs;')

# Pass the token to DuckDB via CREATE SECRET so httpfs can read hf:// paths
if hf_token:
    con.execute("CREATE SECRET (TYPE HUGGINGFACE, TOKEN '" + hf_token + "')")

print('DuckDB connected. Ready to query.')

# Warehouse path helper
WH = 'hf://datasets/FlyRank/internship-warehouse'
FACT = WH + '/fact_content_daily_performance'

Note: you may need to restart the kernel to use updated packages.
HF token loaded.
DuckDB connected. Ready to query.


## 1. The contract (five answers)

### 1a. One row = one content item on one report date

Each row in `fact_content_daily_performance` is one pseudonymized content item, one client, one report date. The grain is (report_date, client_id, content_id). For my lane (Refresh / Content Opportunity Scoring), I care about whether a page's search visibility is declining so an editor can decide which page to rewrite first.

### 1b. Tables I use

- `fact_content_daily_performance` (partitioned by month) for daily search and engagement metrics.
- `dim_content` for content metadata (content_type, word_count, etc.).
- `dim_clients` for client-level context (gsc_data_start, ga4_data_start).

### 1c. Time window

I iterate on `month=2026-03` (March 2026), a mid-panel month. The label uses the last 30 days vs previous 30 days within that month. I treat June 2026 (the final month) as a sealed test month and never develop label logic there.

### 1d. Label

`is_declining_label` = 1 when `trend_direction == 'down'`. A page is declining when its impressions in the last 30 days dropped more than 20% compared to the 30 days before that. This is an observed outcome computed from actual GSC impression data.

### 1e. One thing I deliberately exclude

Product-decision flags (health_score, needs_ctr_fix, etc.). These encode a decision someone already made. Using them as features would be circular: the model would learn the product team's rule, not the underlying signal.

## 2. Field classification

### Features (knowable before the decision moment)
- `gsc_impressions` / `gsc_clicks` / `gsc_avg_position` / `gsc_ctr` — trailing search signals
- `ga4_sessions` / `ga4_engaged_sessions` / `ga4_pageviews` — engagement signals
- `content_age_days` / `days_since_last_update` — freshness signals from dim_content
- `word_count` — content depth from dim_content

### Label
- `trend_direction` / `trend_pct` — source of is_declining_label. NEVER a feature.

### Context (grouping/splitting only)
- `client_id`, `content_id`, `report_date` — for joins, grouped splits, time alignment

### Excluded
- Product flags (health_score, etc.) — product decisions, not observed signals
- `model_used`, `provider_used` — implementation metadata, not page health

## 3. Verify with queries

Every claim above gets a query. A contract claim without a query is a guess.

In [6]:
# Query 1: Grain check
query_grain = """
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as cnt
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING cnt > 1
LIMIT 5
"""
result = con.execute(query_grain).fetchall()
print(f'Grain check: {len(result)} rows with duplicates')
if len(result) == 0:
    print('Grain holds: each (date, client, content) is unique.')
else:
    for r in result:
        print(r)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check: 0 rows with duplicates
Grain holds: each (date, client, content) is unique.


In [7]:
# Query 2: Row count and date span
query_count = """
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as first_date,
    MAX(report_date) as last_date,
    COUNT(DISTINCT client_hash_id) as clients,
    COUNT(DISTINCT content_hash_id) as content_items
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""
result = con.execute(query_count).fetchone()
print(f'Total rows: {result[0]:,}')
print(f'Date range: {result[1]} to {result[2]}')
print(f'Clients: {result[3]}')
print(f'Content items: {result[4]:,}')

Total rows: 9,841,378
Date range: 2026-03-01 to 2026-03-31
Clients: 55
Content items: 331,437


In [8]:
# Query 3: Availability with IS TRUE
query_available = """
SELECT
    COUNT(*) as total,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as gsc_available,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available,
    SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) as both_available
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""
result = con.execute(query_available).fetchone()
print(f'Total rows: {result[0]:,}')
print(f'GSC available (IS TRUE): {result[1]:,} ({result[1]/result[0]:.1%})')
print(f'GA4 available (IS TRUE): {result[2]:,} ({result[2]/result[0]:.1%})')
print(f'Both available: {result[3]:,} ({result[3]/result[0]:.1%})')
print(f'\nRows surviving after IS TRUE filter on both: {result[3]:,}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 9,841,378
GSC available (IS TRUE): 3,611,061 (36.7%)
GA4 available (IS TRUE): 413,966 (4.2%)
Both available: 364,347 (3.7%)

Rows surviving after IS TRUE filter on both: 364,347


## 4. Five features (max)

Built from March 2026 data. Each feature has a one-line justification: why it is knowable at the decision moment.

In [9]:
# Build a 5-feature frame from March 2026
query_features = """
WITH base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        MAX(report_date) as latest_date,
        SUM(gsc_impressions) as impressions_90d,
        SUM(gsc_clicks) as clicks_90d,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) as avg_position,
        SUM(ga4_sessions) as sessions_90d,
        SUM(ga4_engaged_sessions) as engaged_90d
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    client_hash_id,
    content_hash_id,
    impressions_90d,
    clicks_90d,
    avg_position,
    CASE WHEN impressions_90d > 0 THEN clicks_90d * 100.0 / impressions_90d ELSE 0 END as ctr_pct,
    CASE WHEN sessions_90d > 0 THEN engaged_90d * 100.0 / sessions_90d ELSE 0 END as engagement_rate
FROM base
WHERE impressions_90d > 0
"""
features_df = con.execute(query_features).fetchdf()
print(f'Feature frame: {len(features_df)} content items')
features_df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 176738 content items


,client_hash_id,content_hash_id,impressions_90d,clicks_90d,avg_position,ctr_pct,engagement_rate
0,client_62f4a7e64f5e0096,content_c1c03ff14a2081da,818.0,4.0,3.366250,0.488998,0.0
1,client_62f4a7e64f5e0096,content_56f941face90a36b,26452.0,69.0,4.039174,0.260850,0.0
2,client_62f4a7e64f5e0096,content_d22be7adcfd8681c,1302.0,4.0,7.287306,0.307220,0.0
3,client_62f4a7e64f5e0096,content_69787872d970725d,8937.0,28.0,3.758821,0.313304,0.0
4,client_62f4a7e64f5e0096,content_e12e723d3aad35ec,10522.0,16.0,4.874505,0.152062,0.0
5,client_62f4a7e64f5e0096,content_8218632c79b2b721,1352.0,3.0,3.025258,0.221893,0.0
6,client_62f4a7e64f5e0096,content_805997d1c2b1f74f,1622.0,1.0,2.964079,0.061652,0.0
7,client_62f4a7e64f5e0096,content_32071e516777b3d4,5047.0,2.0,6.908624,0.039628,0.0
8,client_62f4a7e64f5e0096,content_b4ee442b8926b21d,1650.0,3.0,2.768462,0.181818,0.0
9,client_62f4a7e64f5e0096,content_2cd4747abf20d0f2,744.0,2.0,4.346273,0.268817,0.0


### Feature justification

1. **impressions_90d** — Knowable because it is trailing GSC data from the past 90 days, fully observed before any decision.
2. **clicks_90d** — Same window as impressions. Observed search behavior, not a prediction.
3. **avg_position** — Mean GSC position over the window. Lower means higher rank. Observable before any refresh decision.
4. **ctr_pct** — Clicks / impressions. Measures snippet quality. Computed from observed data only.
5. **engagement_rate** — Engaged sessions / total sessions. On-page signal from GA4. Observable after a user visits, but before any model prediction.

## 5. The trap: deliberate leakage

I add ONE label-derived column (trend_pct) and watch the score jump toward perfect. Then I delete it and keep the honest number. This is the leakage lesson from notebook 02, performed on real warehouse data.

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

# Build label: last 30 days of March vs previous 30 days (needs Jan-Mar partitions)
query_label = """
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        SUM(gsc_impressions) as impressions
    FROM read_parquet([
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-01/data_0.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    ])
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id, report_date
),
windowed AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date >= '2026-02-18' THEN impressions ELSE 0 END) as last_30d,
        SUM(CASE WHEN report_date >= '2026-01-19' AND report_date < '2026-02-18' THEN impressions ELSE 0 END) as prev_30d
    FROM daily
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    client_hash_id,
    content_hash_id,
    last_30d,
    prev_30d,
    CASE WHEN prev_30d > 0 THEN (last_30d - prev_30d) * 100.0 / prev_30d ELSE 0 END as trend_pct,
    CASE WHEN prev_30d > 0 AND last_30d < prev_30d * 0.8 THEN 1 ELSE 0 END as is_declining_label
FROM windowed
WHERE prev_30d > 0
"""
label_df = con.execute(query_label).fetchdf()
print(f'Label frame: {len(label_df)} items, declining rate: {label_df["is_declining_label"].mean():.3f}')

# Merge features with label
merged = features_df.merge(label_df[['client_hash_id', 'content_hash_id', 'is_declining_label', 'trend_pct']],
                           on=['client_hash_id', 'content_hash_id'], how='inner')
print(f'Merged: {len(merged)} items')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Label frame: 131602 items, declining rate: 0.237
Merged: 111309 items


In [15]:
# Honest model: features only (no leakage)
feature_cols = ['impressions_90d', 'clicks_90d', 'avg_position', 'ctr_pct', 'engagement_rate']
X = merged[feature_cols].fillna(0)
y = merged['is_declining_label']
groups = merged['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=groups))

rf_honest = RandomForestClassifier(n_estimators=100, random_state=42)
rf_honest.fit(X.iloc[tr_idx], y.iloc[tr_idx])
honest_score = rf_honest.score(X.iloc[te_idx], y.iloc[te_idx])
print(f'Honest model accuracy: {honest_score:.3f}')

# Leaky model: add trend_pct (label-derived)
X_leaky = merged[feature_cols + ['trend_pct']].fillna(0)
rf_leaky = RandomForestClassifier(n_estimators=100, random_state=42)
rf_leaky.fit(X_leaky.iloc[tr_idx], y.iloc[tr_idx])
leaky_score = rf_leaky.score(X_leaky.iloc[te_idx], y.iloc[te_idx])
print(f'Leaky model accuracy (with trend_pct): {leaky_score:.3f}')
print(f'\nAccuracy jump: {honest_score:.3f} -> {leaky_score:.3f} (+{leaky_score - honest_score:.3f})')
print('trend_pct IS the label in disguise. Using it is leakage.')
print('The honest number is the one to keep.')

Honest model accuracy: 0.904
Leaky model accuracy (with trend_pct): 1.000

Accuracy jump: 0.904 -> 1.000 (+0.096)
trend_pct IS the label in disguise. Using it is leakage.
The honest number is the one to keep.


## 6. Limitation

One named limitation of this slice: the 90-day aggregation window means I cannot distinguish between a page that declined steadily over three months and one that crashed in the last week. The daily granularity is available in the warehouse, but this contract uses the 90-day rollup. For the capstone, I would use daily-level data to capture the shape of the decline, not just its existence.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.